# MoE speech denoising demo (waveform log-FFT routing)

This notebook demonstrates **end-to-end denoising on real speech** using:

1. **Waveform router** — log-FFT + time features on each **512-sample frame** → RandomForest → noise family  
2. **`MoEHardDenoiser`** — `forward_with_route(noisy_frame, predicted_family)`

**Pipeline:** `.wav` → frames → features → classify → MoE denoise → stitch → output `.wav`

Run from repo root. Requires LibriSpeech under `data/librispeech` (same as other notebooks).

Set `FAST_MODE = True` for a quick run (~5–15 min CPU); `False` for stronger models. Demo clip length: `DEMO_SECONDS = 6` (~6 s of speech).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "moe_baseline").is_dir():
    raise RuntimeError("Run from repo root (must contain moe_baseline/)")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# %pip install -q -r requirements.txt

import joblib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Audio, display
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix

from moe_baseline.config import CHANNEL_NAMES, FRAME_SIZE, NUM_FAMILIES, NOISE_CLF_N_PER_FAMILY, SEED, data_root
from moe_baseline.librispeech import get_train_test_files
from moe_baseline.audio import FrameDataset
from moe_baseline.datasets import NoisyFrameDataset
from moe_baseline.model import MoEHardDenoiser
from moe_baseline.routers import train_demo_router_speech, train_waveform_router
import importlib
import moe_baseline.train as _moe_train
importlib.reload(_moe_train)
from moe_baseline.train import set_seed, train_moe_speech_with_denoising, eval_denoising_oracle
if not hasattr(_moe_train, "train_moe_speech_with_denoising"):
    raise ImportError(
        "train_moe_speech_with_denoising missing — restart kernel and ensure moe_baseline/train.py is saved."
    )
from moe_baseline.demo import (
    build_mixed_noisy_clip,
    choose_demo_clip,
    clip_noise_reduction_db,
    denoise_frames,
    frames_to_waveform,
    predict_routes_with_smoothing,
    routing_accuracy,
    save_wav,
    smooth_route_predictions,
)

FAST_MODE = True
RELOAD_CHECKPOINTS = True  # False to retrain routers + MoE
N_PER_FAMILY = 500 if FAST_MODE else NOISE_CLF_N_PER_FAMILY
DEMO_ROUTER_SAMPLES = 4000 if FAST_MODE else 12000
MOE_EPOCHS = 3 if FAST_MODE else 12
MOE_ROUTER_ONLY_EPOCHS = 2 if FAST_MODE else 5
MOE_MSE_WEIGHT = 0.05 if FAST_MODE else 0.12
MOE_JOINT_EPOCHS = 2 if FAST_MODE else 6
MOE_DENOISE_EPOCHS = 4 if FAST_MODE else 18
MOE_BATCHES_PER_FAMILY = 40 if FAST_MODE else 150
ROUTE_VOTE_WINDOW = 5
DEMO_SECONDS = 6.0

OUT_DIR = REPO_ROOT / "outputs" / "moe_speech_demo"
CKPT_DIR = REPO_ROOT / "checkpoints" / "moe_speech_demo"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| FAST_MODE:", FAST_MODE)
print("Demo:", DEMO_SECONDS, "s | route smoothing window:", ROUTE_VOTE_WINDOW, "| MoE MSE weight:", MOE_MSE_WEIGHT)


## 1. Train routers + MoE (with denoising phase)

- **Routers:** log-FFT RF (pure noise + noisy speech).
- **MoE phase A–B:** light joint routing + small MSE.
- **MoE phase C:** **oracle-route denoising** — each batch uses one noise family; `forward_with_route(noisy, true_family)` vs **clean speech** (MSE + SI-SDR). Each expert learns its own noise type.


In [2]:
train_files, test_files = get_train_test_files(data_root())
train_frame_ds = FrameDataset(train_files[:80] if FAST_MODE else train_files)
train_ds = NoisyFrameDataset(train_frame_ds, "train", np.random.default_rng(SEED + 2))
test_frame_ds = FrameDataset(test_files[:40] if FAST_MODE else test_files)
test_ds = NoisyFrameDataset(test_frame_ds, "val", np.random.default_rng(SEED + 3))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

_ckpt_ok = (
    RELOAD_CHECKPOINTS
    and (CKPT_DIR / "demo_router.joblib").exists()
    and (CKPT_DIR / "moe_denoiser.pt").exists()
)

if _ckpt_ok and (CKPT_DIR / "waveform_router.joblib").exists():
    print("Loading checkpoints from", CKPT_DIR)
    waveform_router = joblib.load(CKPT_DIR / "waveform_router.joblib")
    demo_router = joblib.load(CKPT_DIR / "demo_router.joblib")
    model = MoEHardDenoiser(FRAME_SIZE, NUM_FAMILIES, use_shared_expert=True).to(DEVICE)
    model.load_state_dict(torch.load(CKPT_DIR / "moe_denoiser.pt", map_location=DEVICE))
    denoise_val = eval_denoising_oracle(model, test_loader, DEVICE, max_batches=40)
    print(f"Loaded MoE — oracle val NR {denoise_val['noise_reduction_db']:.2f} dB, MSE {denoise_val['mse']:.6f}")
    acc_pure = acc_demo = None
else:
    _train_router = not (RELOAD_CHECKPOINTS and (CKPT_DIR / "demo_router.joblib").exists())
    if _train_router:
        print("Training waveform router (pure noise)...")
        waveform_router, acc_pure, _ = train_waveform_router(device=DEVICE, n_per_family=N_PER_FAMILY)
        print(f"  Pure-noise holdout: {acc_pure:.4f}")
        print("Training demo router (noisy speech)...")
        demo_router, acc_demo = train_demo_router_speech(train_ds, max_n=DEMO_ROUTER_SAMPLES)
        print(f"  Noisy-speech holdout: {acc_demo:.4f}")
        joblib.dump(waveform_router, CKPT_DIR / "waveform_router.joblib")
        joblib.dump(demo_router, CKPT_DIR / "demo_router.joblib")
    else:
        waveform_router = joblib.load(CKPT_DIR / "waveform_router.joblib")
        demo_router = joblib.load(CKPT_DIR / "demo_router.joblib")
        acc_pure = acc_demo = None
        print("Loaded routers; training MoE only.")

    model = MoEHardDenoiser(FRAME_SIZE, NUM_FAMILIES, use_shared_expert=True).to(DEVICE)
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=True)

    print(f"\nMoE training: joint={MOE_JOINT_EPOCHS} epochs, denoise={MOE_DENOISE_EPOCHS} (oracle family routes)...")
    denoise_metrics = train_moe_speech_with_denoising(
        model,
        train_frame_ds,
        train_loader,
        test_loader,
        DEVICE,
        joint_epochs=MOE_JOINT_EPOCHS,
        denoise_epochs=MOE_DENOISE_EPOCHS,
        router_only_epochs=1 if FAST_MODE else 2,
        mse_weight=MOE_MSE_WEIGHT,
        batches_per_family=MOE_BATCHES_PER_FAMILY,
        use_si_sdr=False,  # set True with si_sdr_weight=0.05 after MSE stabilizes
    )
    torch.save(model.state_dict(), CKPT_DIR / "moe_denoiser.pt")
    print("Saved MoE to", CKPT_DIR / "moe_denoiser.pt")
    print("Oracle denoising on val:", denoise_metrics)


## 2. Build mixed-noise demo clip

One LibriSpeech utterance is split into **6 contiguous segments**; each segment gets a **different** synthetic noise family (oracle labels known for evaluation).

In [3]:
demo_audio, demo_path = choose_demo_clip(test_files, seconds=DEMO_SECONDS)
print("Demo file:", demo_path)
print("Duration (s):", len(demo_audio) / 16000)

clean_frames, noisy_frames, true_labels, segments = build_mixed_noisy_clip(demo_audio, split="val", device=DEVICE)
noisy_audio = frames_to_waveform(noisy_frames)
clean_audio = frames_to_waveform(clean_frames)

print("Frames:", len(noisy_frames))
for name, idx in segments.items():
    if len(idx):
        print(f"  {name:10s} frames {idx[0]}..{idx[-1]} ({len(idx)} frames)")

save_wav(OUT_DIR / "01_clean_speech.wav", clean_audio)
save_wav(OUT_DIR / "02_mixed_noisy.wav", noisy_audio)
print("Wrote", OUT_DIR / "01_clean_speech.wav", "and", OUT_DIR / "02_mixed_noisy.wav")

Demo file: /Users/aaryakhanna/Documents/GitHub/autoencoder_denoiser/data/librispeech/LibriSpeech/test-clean/1089/134686/1089-134686-0000.flac
Duration (s): 4.0
Frames: 125
  hiss       frames 0..20 (21 frames)
  pulse      frames 21..41 (21 frames)
  stepped    frames 42..62 (21 frames)
  warbler    frames 63..83 (21 frames)
  recorded   frames 84..104 (21 frames)
  spark      frames 105..124 (20 frames)
Wrote /Users/aaryakhanna/Documents/GitHub/autoencoder_denoiser/outputs/moe_speech_demo/01_clean_speech.wav and /Users/aaryakhanna/Documents/GitHub/autoencoder_denoiser/outputs/moe_speech_demo/02_mixed_noisy.wav


## 3. Denoise with waveform routing (+ temporal smoothing)

| Output | Routing |
|--------|----------|
| Oracle | True family per segment (upper bound) |
| Demo RF (raw) | Per-frame log-FFT RF on noisy speech |
| Demo RF (smoothed) | Majority vote over `ROUTE_VOTE_WINDOW` frames — **recommended for listening** |
| Fair RF | Pure-noise-trained router (comparison) |


In [4]:
pred_demo_raw, pred_demo_smooth = predict_routes_with_smoothing(
    demo_router, noisy_frames, window=ROUTE_VOTE_WINDOW
)
pred_fair = predict_routes_with_smoothing(waveform_router, noisy_frames, window=1)[0]

denoised_oracle, _ = denoise_frames(noisy_frames, model, demo_router, DEVICE, route_indices=true_labels)
denoised_demo_raw, _ = denoise_frames(noisy_frames, model, demo_router, DEVICE, route_indices=pred_demo_raw)
denoised_demo_smooth, _ = denoise_frames(noisy_frames, model, demo_router, DEVICE, route_indices=pred_demo_smooth)
denoised_fair, _ = denoise_frames(noisy_frames, model, waveform_router, DEVICE, route_indices=pred_fair)

metrics = {
    "oracle_route_acc": routing_accuracy(true_labels, true_labels),
    "demo_rf_raw_acc": routing_accuracy(true_labels, pred_demo_raw),
    "demo_rf_smooth_acc": routing_accuracy(true_labels, pred_demo_smooth),
    "fair_rf_acc": routing_accuracy(true_labels, pred_fair),
    "nr_db_oracle": clip_noise_reduction_db(clean_frames, noisy_frames, denoised_oracle),
    "nr_db_demo_raw": clip_noise_reduction_db(clean_frames, noisy_frames, denoised_demo_raw),
    "nr_db_demo_smooth": clip_noise_reduction_db(clean_frames, noisy_frames, denoised_demo_smooth),
    "nr_db_fair": clip_noise_reduction_db(clean_frames, noisy_frames, denoised_fair),
}

print("=" * 60)
print("ROUTING on 6 s mixed-noise demo (per frame)")
print("=" * 60)
for k, v in metrics.items():
    if k.endswith("_acc"):
        print(f"  {k:<24} {v:.4f}")
print("-" * 60)
print("Noise reduction vs clean (dB, higher=better)")
for k, v in metrics.items():
    if k.startswith("nr_db"):
        print(f"  {k:<24} {v:.2f} dB")
print("=" * 60)
print("\nDemo RF (smoothed) classification report:")
print(classification_report(true_labels, pred_demo_smooth, target_names=CHANNEL_NAMES, zero_division=0))

save_wav(OUT_DIR / "03_denoised_oracle_route.wav", frames_to_waveform(denoised_oracle))
save_wav(OUT_DIR / "04_denoised_demo_raw.wav", frames_to_waveform(denoised_demo_raw))
save_wav(OUT_DIR / "05_denoised_demo_smoothed.wav", frames_to_waveform(denoised_demo_smooth))
save_wav(OUT_DIR / "06_denoised_fair_router.wav", frames_to_waveform(denoised_fair))
print("Saved denoised WAVs to", OUT_DIR)


In [5]:
sr = 16000
n_show = int(min(4.0 * sr, len(clean_audio)))

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
t = np.arange(n_show) / sr
axes[0].plot(t, clean_audio[:n_show], lw=0.7)
axes[0].set_title("Clean speech (reference)")
axes[1].plot(t, noisy_audio[:n_show], lw=0.7, color="C1")
axes[1].set_title("Mixed noise (6 families)")
axes[2].plot(t, frames_to_waveform(denoised_demo_raw)[:n_show], lw=0.7, color="C2")
axes[2].set_title(f"Denoised — demo RF raw (route acc {metrics['demo_rf_raw_acc']:.2f})")
axes[3].plot(t, frames_to_waveform(denoised_demo_smooth)[:n_show], lw=0.7, color="C3")
axes[3].set_title(f"Denoised — demo RF smoothed (route acc {metrics['demo_rf_smooth_acc']:.2f})")
axes[3].set_xlabel("Time (s)")
for ax in axes:
    ax.set_ylabel("Amp")
plt.tight_layout()
plt.savefig(OUT_DIR / "waveforms_comparison.png", dpi=120)
plt.show()

# Routing timeline
fig, ax = plt.subplots(figsize=(12, 3))
ax.scatter(np.arange(len(true_labels)), true_labels, s=10, c="k", label="true", alpha=0.5)
ax.plot(pred_demo_raw, ".", ms=4, alpha=0.5, label="demo raw")
ax.plot(pred_demo_smooth, "x", ms=5, alpha=0.8, label=f"demo smoothed (w={ROUTE_VOTE_WINDOW})")
for name, idx in segments.items():
    if len(idx):
        ax.axvline(idx[0], color="gray", ls=":", lw=0.6)
ax.set_yticks(range(NUM_FAMILIES))
ax.set_yticklabels(CHANNEL_NAMES)
ax.set_xlabel("Frame index")
ax.set_title("Routing over 6 s clip (vertical lines = segment boundaries)")
ax.legend(loc="upper right", ncol=3)
plt.tight_layout()
plt.savefig(OUT_DIR / "routing_timeline.png", dpi=120)
plt.show()


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pred, title in zip(
    axes,
    [pred_demo_raw, pred_demo_smooth],
    ["Demo RF raw", f"Demo RF smoothed (w={ROUTE_VOTE_WINDOW})"],
):
    cm = confusion_matrix(true_labels, pred, labels=list(range(NUM_FAMILIES)))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_FAMILIES))
    ax.set_yticks(range(NUM_FAMILIES))
    ax.set_xticklabels(CHANNEL_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CHANNEL_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{title}\nacc={routing_accuracy(true_labels, pred):.3f}")
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_raw_vs_smoothed.png", dpi=120)
plt.show()

# Summary bar chart
summary_df = pd.DataFrame([
    {"method": "Routing acc — demo raw", "value": metrics["demo_rf_raw_acc"]},
    {"method": "Routing acc — demo smooth", "value": metrics["demo_rf_smooth_acc"]},
    {"method": "Routing acc — fair RF", "value": metrics["fair_rf_acc"]},
    {"method": "NR (dB) — demo raw", "value": metrics["nr_db_demo_raw"] / 20},
    {"method": "NR (dB) — demo smooth", "value": metrics["nr_db_demo_smooth"] / 20},
    {"method": "NR (dB) — oracle route", "value": metrics["nr_db_oracle"] / 20},
])
fig, ax = plt.subplots(figsize=(9, 4))
colors = ["C0", "C0", "C2", "C1", "C1", "C3"]
ax.barh(summary_df["method"], summary_df["value"], color=colors)
ax.set_xlabel("Score (routing acc or NR/20 for comparable scale)")
ax.set_title("6 s demo — routing & denoising summary")
ax.axvline(1 / NUM_FAMILIES, color="gray", ls="--", label="chance routing")
plt.tight_layout()
plt.savefig(OUT_DIR / "summary_metrics.png", dpi=120)
plt.show()

display(summary_df)
summary_df.to_csv(OUT_DIR / "demo_metrics.csv", index=False)
print("Wrote", OUT_DIR / "demo_metrics.csv")


In [7]:
print("Listen (6 s clip):\n")
for label, wav in [
    ("Clean", clean_audio),
    ("Mixed noisy", noisy_audio),
    ("Denoised — demo raw", frames_to_waveform(denoised_demo_raw)),
    ("Denoised — demo smoothed (recommended)", frames_to_waveform(denoised_demo_smooth)),
    ("Denoised — oracle routing", frames_to_waveform(denoised_oracle)),
]:
    print(label)
    display(Audio(wav, rate=sr))


### Example run (6 s clip, smoothed routing)

After running the cells above with saved checkpoints:

| Metric | Value |
|--------|-------|
| Routing acc — demo RF raw | ~0.62 |
| Routing acc — demo RF **smoothed** (w=5) | ~**0.65** |
| Routing acc — fair RF | ~0.47 |
| Output WAV (listen) | `05_denoised_demo_smoothed.wav` |

Smoothing typically **raises** routing accuracy and reduces expert switching clicks. Re-train with `FAST_MODE = False` and `RELOAD_CHECKPOINTS = False` for stronger denoising.


## 4. Optional: your own WAV file

Set `CUSTOM_WAV` to a path (16 kHz mono preferred). The notebook will add **one** noise family to the whole clip for a quick test.

In [8]:
CUSTOM_WAV = None  # e.g. REPO_ROOT / "my_recording.wav"
CUSTOM_NOISE_FAMILY = 0  # 0=hiss .. 5=spark

if CUSTOM_WAV is not None:
    from moe_baseline.audio import load_wav, frame_audio
    from moe_baseline.noise import apply_family_noise

    custom = load_wav(Path(CUSTOM_WAV))
    cfs = frame_audio(custom)
    noisy_list = []
    for i in range(len(cfs)):
        c = torch.from_numpy(cfs[i])
        noisy_list.append(apply_family_noise(c, CUSTOM_NOISE_FAMILY, "val", device=DEVICE).cpu().numpy())
    noisy_cf = np.stack(noisy_list)
    pred = predict_waveform_route_batch(demo_router, noisy_cf)
    out_cf, _ = denoise_frames(noisy_cf, model, demo_router, DEVICE, route_indices=pred)
    out_audio = frames_to_waveform(out_cf)
    save_wav(OUT_DIR / "06_custom_denoised.wav", out_audio)
    print("Predicted family:", CHANNEL_NAMES[int(np.median(pred))])
    display(Audio(out_audio, rate=16000))
else:
    print("Set CUSTOM_WAV to run a custom file.")

Set CUSTOM_WAV to run a custom file.


### Outputs

| File | Description |
|------|-------------|
| `01_clean_speech.wav` | Original demo speech (6 s) |
| `02_mixed_noisy.wav` | Six-family mixed noise |
| `04_denoised_demo_raw.wav` | Denoised, per-frame RF |
| **`05_denoised_demo_smoothed.wav`** | **Recommended** — RF + majority vote |
| `03_denoised_oracle_route.wav` | Upper bound (true family) |
| `demo_metrics.csv` | Routing accuracy + noise-reduction table |
| `*.png` | Waveforms, routing timeline, confusion matrices |

Set `RELOAD_CHECKPOINTS = True` after the first full training run.


## Further improvements

- `FAST_MODE = False` for full training (already uses balanced RF + higher MSE when False).
- Increase `ROUTE_VOTE_WINDOW` (7–9) if routing still flickers at segment edges.
- Babble / live mic via `CUSTOM_WAV` cell below.
- Compare to `outputs/classifier_adaptive_reconstruction.wav` from the adaptive notebook.
